# KIET Flood Dataset v1.0 — Colab generation pipeline

Large-scale synthetic flood-scenario generation. The local PC handles development/debugging;
Colab does the heavy generation. All physics code lives in `simulation/` (diffusive-wave
surface + synthetic Manning-pipe drainage). **All drainage is SYNTHETIC (`verified=false`) —
never real KIET infrastructure.**

Resume: re-running the generator skips every `scenario_id` already marked valid/quarantined
in `dataset_manifest.csv`. Deterministic IDs + seeds (master seed 26085).

## 0. Upload the simulation tarball

Built on the local PC (only the files the simulator needs):

```bash
tar czf /tmp/opencode/drishti_sim.tar.gz simulation dataset config tests \
  data/terrain_grid.json data/campus_accurate.geojson data/campus.geojson \
  data/roads.geojson kiet_terrain/campus_osm.geojson
```

Or via CLI: `colab upload -s drishti-gen /tmp/opencode/drishti_sim.tar.gz drishti_sim.tar.gz`

In [ ]:
import shutil
shutil.unpack_archive('/drishti_sim.tar.gz', '/content/drishti')  # CLI upload lands at /
# -- notebook upload alternative: --
# from google.colab import files; files.upload()  # then unpack likewise
%cd /content/drishti
!ls

In [ ]:
# Install deps (numpy stock on Colab images)
!pip install -q h5py pyyaml scipy matplotlib

## 1. Benchmark (record CPU/RAM/disk/speed before sizing the run)

In [ ]:
import os, time, shutil
print('cpu:', os.cpu_count())
print(open('/proc/meminfo').readline().strip())
t, u, f = shutil.disk_usage('/content')
print(f'disk free={f/1e9:.0f}G')
import yaml
from simulation.terrain.twin import Twin
from simulation.drainage.network import generate as gen_net
from simulation.scenarios.suite_v2 import make_prod_suite
from simulation.hydraulics.simulate import simulate
ter = yaml.safe_load(open('config/terrain.yaml'))
dra = yaml.safe_load(open('config/drainage.yaml'))
rain = yaml.safe_load(open('config/rainfall.yaml'))
hyd = yaml.safe_load(open('config/hydraulics.yaml'))
t0=time.time(); twin=Twin(ter); print(f'Twin {time.time()-t0:.1f}s')
net=gen_net(twin,dra,seed=26085,variant=0)
for sp in make_prod_suite(3, 26085):
    t0=time.time(); res=simulate(twin,net,sp,hyd,rain); dt=time.time()-t0
    print(f"nt={res['depth'].shape[0]} sim={dt:.1f}s ({dt/res['depth'].shape[0]:.2f}s/step) maxd={res['max_depth'].max():.3f}m")

## 2. Full v1.0 generation (background, resumable)

Reference sizing (measured 2026-09-06, 2-CPU Colab, 0.57 s/step): 240 prod + 36 OOD,
2 workers → ~100 min, ~500 MB. Scale `--prod-n` up once validated.

In [ ]:
# Run in background so the cell returns immediately; poll the log instead.
import subprocess, sys
log = open('/content/drishti/gen_v1.log', 'ab', buffering=0)
p = subprocess.Popen([sys.executable, '-u', '-m', 'dataset.generator.run_v2',
                      '--prod-n', '240', '--ood-n', '36', '--workers', '2'],
                     stdout=log, stderr=subprocess.STDOUT,
                     start_new_session=True, cwd='/content/drishti')
print('launched pid:', p.pid)

In [ ]:
# Poll progress (re-run this cell any time; Colab may idle-disconnect, generation resumes)
!tail -5 /content/drishti/gen_v1.log
!echo '--- manifest ---'; tail -2 /content/drishti/outputs/datasets/dataset_manifest.csv | cut -c1-200

## 3. Validate + package (after the log shows `done:`)

In [ ]:
!cd /content/drishti && python3 -m dataset.stats --manifest outputs/datasets/dataset_manifest.csv
!cd /content/drishti && python3 -m dataset.qc_plots --manifest outputs/datasets/dataset_manifest.csv
!cd /content/drishti/outputs/datasets && sha256sum kiet_flood_*.h5 ood/kiet_flood_ood.h5 dataset_manifest.csv > SHA256SUMS && cat SHA256SUMS

## 4. Download to local PC

Transfer ONLY the finalized files (never intermediate state):
`kiet_flood_{train,val,test}.h5`, `ood/kiet_flood_ood.h5`, `kiet_flood_quarantine.h5`,
`dataset_manifest.csv`, `kiet_networks_v1.json`, `SHA256SUMS`.

Via CLI (absolute remote paths, verified 2026-09-06):
`colab download -s drishti-gen /content/drishti/outputs/datasets/kiet_flood_train.h5 outputs/datasets/`
(repeat per file). Or mount Drive and copy there first for large files.